First let's clone the cuOpt repository

In [ ]:
!git clone https://github.com/NVIDIA/cuopt.git
%cd cuopt

fatal: destination path 'cuopt' already exists and is not an empty directory.
/home/vmostovoi/nvidia/cuopt


Now we create an environment to allow cuopt to run

In [ ]:
!mamba env create -n cuopt_env -y -f conda/environments/all_cuda-133_arch-x86_64.yaml

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
rapidsai/linux-64                                           Using cache
rapidsai/noarch                                             Using cache
[+] 0.0s
[+] 0.1s
rapidsai-nightly/linux-64 ━━━━━━━━╸━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.0s
rapidsai-nightly/noarch   ━━━━━╸━━━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.0s[+] 0.2s
rapidsai-nightly/linux-64 ━━━━━━━━╸━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.1s
rapidsai-nightly/noarch   ━━━━━╸━━━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.1s

Transaction

  Prefix: /home/vmostovoi/software/miniforge3/envs/cuopt_env

  Updating specs:

   - breathe
   - bzip2
   - c-ares
   - c-compiler
   - ccache
   - clang-tools=20.1.8
   - clang==20.1.8
   - cmake>=4.0
   - cpp-argparse
   - cuda-nvcc
   - cuda-nvtx-dev
   - cuda-sanitizer-api
   - cuda-version=13.3
   - cudf[version="=26.10,>=0.0.0a0"]
   - cu

We build the solver

In [12]:
!mamba run -n cuopt_env      \
    ./build.sh libcuopt      \
    --skip-c-python-adapters \
    --skip-fatbin-write      \
    --skip-tests-build

Building for the architecture of the GPU in the system...
CMake Warning (author) at /home/vmostovoi/software/miniforge3/envs/cuopt_env/share/cmake-4.4/Modules/GNUInstallDirs.cmake:448 (message):
  Unable to determine default CMAKE_INSTALL_LIBDIR directory because no
  target architecture is known.  Please enable at least one language before
  including GNUInstallDirs.
Call Stack (most recent call first):
  /home/vmostovoi/software/miniforge3/envs/cuopt_env/share/cmake-4.4/Modules/GNUInstallDirs.cmake:290 (_GNUInstallDirs_LIBDIR_get_default)
  /home/vmostovoi/software/miniforge3/envs/cuopt_env/share/cmake-4.4/Modules/GNUInstallDirs.cmake:290 (cmake_language)
  /home/vmostovoi/software/miniforge3/envs/cuopt_env/share/cmake-4.4/Modules/GNUInstallDirs.cmake:547 (_GNUInstallDirs_cache_path)
  CMakeLists.txt:11 (include)
This warning is for project developers.  Use -Wno-author to suppress it.

CMake Warning (author) at /home/vmostovoi/software/miniforge3/envs/cuopt_env/share/cmake-4.4/Module

And we ensure we can solve a basic instance

In [13]:
!mamba run -n cuopt_env \
./cpp/build/cuopt_cli datasets/linear_programming/afiro_original.mps

Setting CUDA_MODULE_LOADING to EAGER
Reading file afiro_original.mps
Read file afiro_original.mps in 0.00 seconds
cuOpt version: 26.10.0, git hash: 68a12336, host arch: x86_64, device archs: 120a-real
CPU: Intel(R) Core(TM) Ultra 7 265H, threads: 16C/16T, RAM usage: 10.63/62.26GiB
CPU SIMD target: AVX2
CUDA 13.3, device: NVIDIA RTX PRO 3000 Blackwell Generation Laptop GPU (ID 0), VRAM: 11.50 GiB
CUDA device UUID: d9dd97a0-3eac-dc42-fe23-bec0daa9ac94
Solving a problem with 27 constraints, 32 variables (0 integers), and 83 nonzeros
Problem scaling:
Objective coefficents range:          [3e-01, 1e+01]
Constraint matrix coefficients range: [1e-01, 2e+00]
Constraint rhs / bounds range:        [0e+00, 5e+02]
Variable bounds range:                [0e+00, 0e+00]

Using PSLP presolver
PSLP Presolved problem: 14 constraints, 18 variables, 40 non-zeros
PSLP presolve time: 0.00s
Objective offset 0.000000 scaling_factor 1.000000
Running concurrent (showing only PDLP log)

Dual simplex finished in 0

## Multi-GPU solves

Let's try again to solve `afiro_original`, but this time with 4 gpus, using `--num-gpus 4` and `--method 1`

In [14]:
!mamba run -n cuopt_env \
    ./cpp/build/cuopt_cli datasets/linear_programming/afiro_original.mps \
    --num-gpus 4                                                         \
    --method 1
